# 5.4 — Directional locomotion on `myofullbody` with a myoSuperMimic teacher

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MyoHub/myosuite/blob/dev/tutorials/5.4_Directional_Locomotion.ipynb)

Distil a muscle-driven full-body walker that walks in any commanded direction, on myo_sim's `myofullbody` (the
body of the MyoChallenge 2026 ChaseTag game).

1. **Data and teacher.** A myoSuperMimic tracker (`training/supermimic`) trained on `myoMimic-v0`'s default clips,
   MuscleMimic's 972 KIT training motions retargeted onto `myofullbody`. The steered reference of
   `training/chasetag_teacher` turns it into a teacher that walks any requested heading, by motion matching over the
   KIT locomotion clips of the same list (tutorial 5.7 plays it in the ChaseTag game).
2. **Train.** DAgger distils the teacher into a 528 → 354 MLP student. The student reads only its own body and the
   heading. The Colab-sized run here is short; the published walker came from the full recipe.
3. **Evaluate** with this tutorial's protocol: 8 compass headings × 10 seeds × 200 control steps (2 s) from
   standing, facing east. Reported are survival, mean steps alive and alignment of the displacement with the heading.
4. **Render** the eight headings, and save the walker for [5.6](./5.6_ChaseTag_MyoChallenge.ipynb), which exports it
   as a ChaseTag player.

Runtime: **CPU is enough** (Colab CPU: the quick path takes a few minutes, the full Colab training about an hour).
`QUICK_MODE` is the default; set `MYOSUITE_FULL_DAGGER=1` for the longer training and the full evaluation.

**This tutorial changed.** It used to behaviour-clone MuscleMimic's `mm-10m-2` on the `musclemimic_models` body
(`myoFullBodyDirectional-v0`). It now runs on `myofullbody`, and the walker's 528-d observation keeps the old slots but
gives the heading in the walker's own yaw frame. Walkers of the earlier version (`policy_bc_best.pt`,
`policy_heading_table.pt`) do not load here.

**Data.** Training needs the KIT clips retargeted onto `myofullbody`. Use one of:

- a local folder in `$MYOSUITE_KIT_ROOT`, or the local retarget cache (`scripts/mimic/retarget_amass.py` fills it from
  your AMASS download);
- a Hugging Face dataset of retargeted clips: put its id in `MIMIC_DATASET` in the setup cell.

Without clips, the notebook skips training and evaluates the published walker. The tracker and the walker are
downloaded from `myohub/myosuite-3-baselines`. Set `$MYOSUITE_BASELINES_DIR` to a local folder laid out like that repo to use local files instead (`download_baseline_file`).

> **Citation and data:** SuperMimic (Liao et al., [arXiv:2610.28480](https://arxiv.org/abs/2610.28480)) through
> MyoSuite's port `training/supermimic`; DAgger (Ross, Gordon and Bagnell, AISTATS 2011); MuscleMimic's KIT split
> (Li et al., 2026, [arXiv:2603.25544](https://arxiv.org/abs/2603.25544)). The motions come from
> [AMASS](https://amass.is.tue.mpg.de) (Mahmood et al., ICCV 2019,
> [arXiv:1904.03278](https://arxiv.org/abs/1904.03278)) and the KIT whole-body human motion database (Mandery et al.,
> [2015](https://h2t.iar.kit.edu/pdf/Mandery2015a.pdf), [2016](https://doi.org/10.1109/TRO.2016.2572685)).
> AMASS-derived clips and models trained on them are for non-commercial research under the AMASS license.
> All references: the *References* page of the MyoSuite documentation.

## 5.4.0 — Install

In [1]:
import os
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning)  # mujoco-warp notes on import

# Hugging Face dataset of KIT clips retargeted onto myofullbody (laid out as <model>/<version>/KIT/...).
# Leave empty to use $MYOSUITE_KIT_ROOT or the local retarget cache instead.
MIMIC_DATASET = os.environ.get("MYOSUITE_MIMIC_DATASET", "")  # <- fill in the dataset id
if MIMIC_DATASET:
    os.environ["MYOSUITE_MIMIC_DATASET"] = MIMIC_DATASET

QUICK_MODE = os.environ.get("MYOSUITE_FULL_DAGGER", "0") != "1"
GIT_REF = os.environ.get("MYOSUITE_GIT_REF", "dev")
os.environ["CHASETAG_TEACHER_CACHE"] = os.environ.get(
    "CHASETAG_TEACHER_CACHE", str(Path.home() / ".cache" / "myosuite" / "chasetag_teacher")
)
if sys.platform != "darwin":
    os.environ.setdefault("MUJOCO_GL", "egl")
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
if "google.colab" in sys.modules:
    subprocess.run(["apt-get", "-qq", "install", "-y", "libegl1", "libgles2"], check=False)

try:
    import chasetag_teacher  # noqa: F401
except ImportError:
    # MyoSuite, myoSuperMimic and the teacher project, all from the same branch.
    url = "git+https://github.com/MyoHub/myosuite.git@" + GIT_REF
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        f"myosuite[mjlab] @ {url}",
        f"myosuite-supermimic @ {url}#subdirectory=training/supermimic",
        f"chasetag-teacher @ {url}#subdirectory=training/chasetag_teacher",
        "huggingface_hub", "imageio[ffmpeg]",
    ])

RUN_DIR = Path("runs") / "directional_myofullbody"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("QUICK_MODE", QUICK_MODE)

QUICK_MODE False


## 5.4.1 — Data and teacher

The tracker checkpoint (myoSuperMimic, 30-minute run on `myoMimic-v0`'s default KIT clips) and the steering library:
the KIT locomotion clips of the same list (walks, turns, circles; 860 clips).

In [2]:
from myosuite.core.hf_io import download_baseline_file

# Verified snapshot: tracker, directional walker and ChaseTag self-play player.
BASELINES_REVISION = "7ab693c740a9dd5a812ea5c1d6dcc52e2baa3e9b"
from myosuite.envs.myo.tasks.mimic.tracking import MissingMotionData, kit_root

from chasetag_teacher.library import load_library

# Your own tracker (a training/supermimic checkpoint on myofullbody) via $CHASETAG_TEACHER_CHECKPOINT, else the published one.
TRACKER = os.environ.get("CHASETAG_TEACHER_CHECKPOINT") or download_baseline_file(
    "supermimic/tracker_myofullbody_kit_30min.pt",
    revision=BASELINES_REVISION, fallback_to_main=False,
)
print("tracker:", Path(TRACKER).name if TRACKER else None)

lib = None
try:
    KIT_ROOT = kit_root("myofullbody")  # $MYOSUITE_KIT_ROOT, the local cache, else the dataset
    lib = load_library(str(KIT_ROOT))  # built once (seconds to minutes), then cached
    print(f"steering library: {len(lib.names)} KIT locomotion clips, {lib.m.num_frames} frames")
except MissingMotionData as exc:
    print("[SKIP] no retargeted KIT clips, training is skipped:", exc)
HAVE_TEACHER = lib is not None and TRACKER is not None

tracker: tracker_myofullbody_kit_30min.pt
steering library: 860 KIT locomotion clips, 241643 frames


## 5.4.2 — Scene, observation and the teacher

`DirectionalScene` is one `myofullbody` on flat ground (the scene of `myoMimic-v0`), stepped at 100 Hz like the
ChaseTag players. The walker's **528-d directional observation** (`chasetag_teacher.directional.directional_obs`):

| Slice | Size | Content |
|---|---:|---|
| `0:82` | 82 | joint positions (`qpos[7:]`) |
| `82:164` | 82 | joint velocities (`qvel[6:]`) |
| `164:518` | 354 | muscle activations |
| `518:520` | 2 | root xy velocity in the yaw frame |
| `520:522` | 2 | **commanded heading in the yaw frame**, `[cos(θ − yaw), sin(θ − yaw)]` |
| `522:528` | 6 | roll, pitch, root angular velocity (body frame), root z velocity |

Nothing in it changes when the whole scene turns about the vertical axis, so the walker steers from the relative
heading alone. The earlier version gave the heading in world axes and no yaw, so its walker could not know which way
to turn.

The teacher (`HeadingTeacher`) walks the commanded heading at 1.2 m/s. Here it starts facing east and is sent north.

In [3]:
import math

import numpy as np

from chasetag_teacher.directional import COMPASS, DirectionalScene, HeadingTeacher

scene = DirectionalScene()
obs = None
if HAVE_TEACHER:
    teacher = HeadingTeacher(str(TRACKER), lib, scene)
    scene.reset(0.0, np.random.default_rng(0))
    teacher.reset(math.radians(COMPASS["N"]))
    start = scene.data.qpos[:2].copy()
    for t in range(300):  # 3 s
        scene.step(teacher.act())
        if scene.fallen:
            break
    obs = scene.obs(teacher.theta)
    d = scene.data.qpos[:2] - start
    print(f"teacher, 3 s heading N: fallen={scene.fallen}, moved {np.linalg.norm(d):.2f} m toward "
          f"{math.degrees(math.atan2(d[1], d[0])):.0f} deg, now facing {math.degrees(scene.facing):.0f} deg")
    print("observation", obs.shape, "heading slot", obs[520:522].round(3))
else:
    print("[SKIP] no teacher (needs clips and the tracker)")

teacher, 3 s heading N: fallen=False, moved 2.57 m toward 67 deg, now facing 74 deg
observation (528,) heading slot [ 0.263 -0.965]


## 5.4.3 — Train the student (DAgger)

Each round plays episodes from standing with a random facing, a random heading and one or two heading changes.
Round 0 lets the teacher act. Later rounds let the previous student act with probability 1 − β; the teacher then
takes over once the pelvis drops below 0.8 m. Every visited state is labelled with the teacher's excitations, and the
student (`ActorCritic`, 512 × 6, input normalisation folded into the first layer) is refitted by MSE on all rounds.

`QUICK_MODE`: 2 rounds × 4 episodes of 4 s (minutes on CPU; the student will still fall).
`MYOSUITE_FULL_DAGGER=1`: 4 rounds × 48 episodes of 10 s (about an hour on Colab CPU).
The published walker came from `training/chasetag_teacher/scripts/dagger_directional.py` (480 episodes per
round, 12 processes; the student of round 5, the best on this protocol); see the table in 5.4.4.

The outputs stored in this notebook come from a run with `MYOSUITE_FULL_DAGGER=1`.

In [4]:
import torch

from chasetag_teacher.directional import DIR_OBS_DIM, EpisodeCfg, WalkerNet, collect_episode, torch_policy
from chasetag_teacher.distill import train
from myosuite.envs.myo.tasks.mimic.policy import ActorCritic

ROUNDS, EPISODES, STEPS, EPOCHS = (2, 4, 400, 5) if QUICK_MODE else (4, 48, 1000, 20)
BETAS = [1.0, 0.3, 0.1, 0.0]
MY_WALKER = RUN_DIR / "walker_myofullbody.pt"

my_walker = None
if HAVE_TEACHER:
    X, Y, init = [], [], None
    for r in range(ROUNDS):
        beta = BETAS[min(r, len(BETAS) - 1)]
        student = torch_policy(my_walker) if my_walker is not None else None
        falls = 0
        for i in range(EPISODES):
            cfg = EpisodeCfg(steps=STEPS, takeover_z=0.8 if i % 2 == 0 else 0.0)
            ep = collect_episode(scene, teacher, student, beta, seed=1000 * r + i, cfg=cfg)
            X.append(ep["obs"])
            Y.append(ep["act"])
            falls += ep["student"] and ep["fell"] is not None
        res = train(np.concatenate(X), np.concatenate(Y), epochs=EPOCHS, batch=512, init=init,
                    obs_dim=DIR_OBS_DIM, log=lambda s: None)
        init = res
        net = ActorCritic(DIR_OBS_DIM, 354)
        net.load_state_dict(res["state"])
        my_walker = WalkerNet(net).eval()
        print(f"round {r}: beta {beta}, {sum(len(x) for x in X)} samples, val MSE {res['val_mse']:.4f}, "
              f"student falls {falls}")
    torch.save(res["state"], MY_WALKER)
    print("saved", MY_WALKER)
else:
    print("[SKIP] training needs the teacher")

round 0: beta 1.0, 18611 samples, val MSE 0.0264, student falls 0


round 1: beta 0.3, 26833 samples, val MSE 0.0232, student falls 29


round 2: beta 0.1, 32378 samples, val MSE 0.0231, student falls 42


round 3: beta 0.0, 36178 samples, val MSE 0.0241, student falls 48
saved runs/directional_myofullbody/walker_myofullbody.pt


## 5.4.4 — Evaluate

The published walker (`directional/walker_myofullbody.pt`, the full recipe) and, if you trained one above, yours, with
this tutorial's protocol. Each rollout starts standing and facing east, so north and south need a 90° turn and west a
180° turn within the 2 s.

Results of the published walker with the full protocol (8 headings × 10 seeds), next to the walker this tutorial
shipped before (`policy_heading_table.pt` on the `musclemimic_models` body, `myoFullBodyDirectional-v0`, from that
env's own mid-gait reset):

| Walker | Body / env | Survival@200 | Mean steps alive | Alignment | Survival over 10 s |
|---|---|---:|---:|---:|---:|
| published (`walker_myofullbody.pt`) | `myofullbody`, this scene, from standing | 91 % | 195.7 | +0.85 | 85 % (alignment +0.93) |
| earlier `policy_heading_table.pt` | `musclemimic_models`, `myoFullBodyDirectional-v0`, mid-gait start | 34 % | 168.2 | +0.92 | 0 % |

Per heading (published walker, 200 steps): E 100 %, NE 100 %, N 90 %, NW 100 %, W 90 %, SW 70 %, S 80 %, SE 100 %.
As a ChaseTag player (tutorial 5.6's export, 48 episodes per match and side) it tags a player that does nothing in
100 % of the bouts and wins 64 % of its bouts against the FBVs baseline; the earlier walker tagged 0 % and won 6 %.


Behaviour cloning and DAgger can fit the teacher closely and still fall; judge a walker by these closed-loop
numbers.

In [5]:
from chasetag_teacher.directional import evaluate, load_walker

LABELS = ["N", "E", "W"] if QUICK_MODE else list(COMPASS)
SEEDS = 2 if QUICK_MODE else 10

PUBLISHED = download_baseline_file("directional/walker_myofullbody.pt", revision=BASELINES_REVISION, fallback_to_main=False)
walkers = {"published": load_walker(PUBLISHED)}
if my_walker is not None:
    walkers["yours"] = my_walker


def report(name, res):
    print(f"{name}:  {'dir':>3}  {'survival':>8}  {'steps alive':>11}  {'alignment':>9}")
    for lab, r in res.items():
        print(f"{'':{len(name) + 1}}  {lab:>3}  {r.survival:>8.0%}  {r.steps_alive:>11.1f}  {r.alignment:>+9.3f}")


results = {}
for name, net in walkers.items():
    results[name] = evaluate(torch_policy(net), LABELS, seeds=SEEDS, steps=200, scene=scene)
    report(name, results[name])

published:  dir  survival  steps alive  alignment
              E      100%        200.0     +0.964
             NE      100%        200.0     +0.619
              N       90%        193.3     +0.944
             NW      100%        200.0     +0.797
              W       90%        196.0     +0.586
             SW       70%        190.9     +0.974
              S       80%        185.3     +0.952
             SE      100%        200.0     +0.944


yours:  dir  survival  steps alive  alignment
          E       30%        173.8     +0.769
         NE        0%        144.3     +0.697
          N        0%        131.9     +0.889
         NW        0%        128.3     +0.757
          W       10%        157.0     +0.546
         SW        0%        148.1     +0.597
          S        0%        166.7     +0.421
         SE       40%        178.6     +0.152


## 5.4.5 — Render the compass directions

The published walker, one panel per heading, 2 s each, written as MP4 and shown inline.

In [6]:
import imageio
import mujoco
from IPython.display import Video, display

from chasetag_teacher.directional import rollout

RENDER_LABELS = ["N", "E"] if QUICK_MODE else list(COMPASS)
W, H = 320, 240
policy = torch_policy(walkers["published"])
renderer = mujoco.Renderer(scene.model, height=H, width=W)
cam = mujoco.MjvCamera()
cam.type = mujoco.mjtCamera.mjCAMERA_FREE
cam.distance, cam.azimuth, cam.elevation = 3.5, 135.0, -20.0

panels = []
for lab in RENDER_LABELS:
    frames = []

    def record(s, frames=frames):
        cam.lookat[:] = [s.data.qpos[0], s.data.qpos[1], 0.9]
        renderer.update_scene(s.data, camera=cam)
        frames.append(renderer.render().copy())

    alive, n, align = rollout(scene, policy, math.radians(COMPASS[lab]), seed=2, steps=200, record=record)
    frames += [frames[-1]] * (200 - len(frames))
    panels.append(frames)
    print(f"{lab}: alive={alive} steps={n} alignment={align:+.2f}")
renderer.close()

cols = min(4, len(panels))
blank = np.zeros_like(panels[0][0])
video = RUN_DIR / ("compass_quick.mp4" if QUICK_MODE else "compass_8dirs.mp4")
with imageio.get_writer(str(video), fps=round(1 / scene.dt), quality=8, macro_block_size=1) as w:  # real time
    for t in range(200):
        tiles = [p[t] for p in panels] + [blank] * (-len(panels) % cols)
        rows = [np.concatenate(tiles[i:i + cols], axis=1) for i in range(0, len(tiles), cols)]
        w.append_data(np.concatenate(rows, axis=0))
display(Video(str(video), embed=True, html_attributes="controls loop"))

E: alive=True steps=200 alignment=+1.00


NE: alive=True steps=200 alignment=+1.00


N: alive=True steps=200 alignment=+0.91


NW: alive=True steps=200 alignment=+0.87


W: alive=True steps=200 alignment=+0.68


SW: alive=True steps=200 alignment=+0.98


S: alive=True steps=200 alignment=+1.00


SE: alive=True steps=200 alignment=+0.97


## 5.4.6 — Use the walker in ChaseTag

`export_chasetag_onnx` wraps a walker as a scored-game player (546 → 354). The heading points at the opponent, or away
from it as runner, turned off the fence and rotated into the walker's yaw frame. Tutorial
[5.6](./5.6_ChaseTag_MyoChallenge.ipynb) does this with your `walker_myofullbody.pt` (upload it into 5.6's Colab) or
with the published walker, and submits it.

In [7]:
from chasetag_teacher.directional import export_chasetag_onnx

source = MY_WALKER if (MY_WALKER.is_file() and not QUICK_MODE) else PUBLISHED
onnx_path = export_chasetag_onnx(source, RUN_DIR / "chasetag_walker.onnx")
print("ChaseTag player from", "your walker" if source == MY_WALKER else "the published walker", "->", onnx_path)

ChaseTag player from your walker -> runs/directional_myofullbody/chasetag_walker.onnx


## Notes

- `chasetag_teacher.directional` holds the scene, the observation, the teacher wrapper, DAgger episodes, the
  evaluation and the ChaseTag export; `scripts/dagger_directional.py` is the multi-process recipe of the published
  walker (`training/chasetag_teacher/README.md`).
- The walker acts at 100 Hz; the tracker of the teacher acts at 50 Hz and holds its excitation.
- The evaluation starts from standing (`qpos0` with small joint noise) rather than from a motion clip, so it needs no
  clip; the ChaseTag players start the same way.